In [9]:
# ========== 导入：把后面要用的工具箱搬进来 ==========
# 若导入失败：先确认已在命令行激活课程环境（例如 conda/venv 里的 llms），再重跑本格

# 导入标准库 os：读环境变量（Environment Variables），例如 OPENAI_API_KEY
import os
# 导入标准库 requests：用 HTTP GET 抓取患者教育网站页面
import requests
# 导入标准库 json：读写缓存文件 brochure_cache.json（字典 ↔ JSON 文本）
import json
# 从 typing 导入 List：给列表做类型标注（Type Hint），方便阅读
from typing import List
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 从 bs4 导入 BeautifulSoup：解析 HTML，提取标题、正文、链接
from bs4 import BeautifulSoup
# 从 IPython.display 导入展示工具：在笔记本里显示 Markdown / 流式更新（本练习后半也可扩展用）
from IPython.display import Markdown, display, update_display
# 从 openai 导入 OpenAI 客户端类：调用云端 Chat Completions API
from openai import OpenAI


In [10]:
# 导入 pandas：表格数据处理库（本笔记本后续若扩展导出表格可用；当前主流程可不依赖它）
import pandas as pd


In [11]:
# ========== 初始化：读 API Key + 选定模型 + 创建客户端 ==========

# 加载 .env：override=True 表示用文件里的值覆盖已有同名环境变量
load_dotenv(override=True)
# 从环境变量读取 OpenAI API 密钥（不要把真实 key 写进笔记本）
api_key = os.getenv('OPENAI_API_KEY')

# 粗检密钥形态：是否存在、是否以 sk-proj- 开头、长度是否看起来合理（不是严格校验）
if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    # 打印提示保留英文：这是运行时给用户看的状态文案，原样不动
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")

# 本练习用的云端小模型名（后面 summarize 里也有硬编码 model= 字符串，两处保持原样）
MODEL = 'gpt-4o-mini'
# 创建 OpenAI 客户端：默认会再从环境变量读 OPENAI_API_KEY
openai = OpenAI()


API key looks good so far


In [12]:
# ========== Website 类：把「一个 URL」变成「标题 + 纯文本 + 链接列表」==========

# 有些网站会拦截「不像浏览器」的请求；带上常见 Chrome User-Agent 更不容易被拒
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class Website:
    """抓取单个网页：保存原始 HTML、清洗后的正文、以及页面上的链接。"""

    def __init__(self, url):
        # 记住当前页面地址，后面写宣传册 / 缓存时都要用
        self.url = url
        # GET 请求下载页面；headers 模拟浏览器
        response = requests.get(url, headers=headers)
        # 原始响应体（字节）：交给 BeautifulSoup 解析
        self.body = response.content
        # 用 html.parser 解析 DOM
        soup = BeautifulSoup(self.body, 'html.parser')
        # 页面 <title>；若缺失则用占位英文（发给模型时仍是英文结构）
        self.title = soup.title.string if soup.title else "No title found"

        # 有 <body> 才抽正文；先删掉 script/style/img/input 等对摘要无用的节点
        if soup.body:
            for irrelevant in soup.body(["script", "style", "img", "input"]):
                # decompose：从树中彻底移除该节点
                irrelevant.decompose()
            # 抽出可见文本：换行分隔、去掉首尾空白
            self.text = soup.body.get_text(separator="\n", strip=True)
        else:
            # 没有 body（少见）→ 空字符串，避免后面报错
            self.text = ""

        # 收集所有 <a href=...>；get('href') 可能为 None
        links = [link.get('href') for link in soup.find_all('a')]
        # 过滤掉空 / None，只保留真有地址的链接
        self.links = [link for link in links if link]

    def get_contents(self):
        # 拼成「标题 + 正文」大字符串，方便塞进 LLM prompt（英文标签保留，影响模型理解格式）
        return f"Webpage Title:\n{self.title}\nWebpage Contents:\n{self.text}\n\n"


In [15]:
# ========== 从 Thuisarts「主题总览」页收集疑似病症页面链接 ==========

def get_condition_links_from_topics_page():
    """抓取荷兰患者教育站 thuisarts.nl 的主题列表，粗筛出病症相关 URL。"""
    # 主题总览页固定地址（不要改：改了就抓不到同一批链接）
    topics_url = "https://www.thuisarts.nl/overzicht/onderwerpen"
    # 带浏览器头 GET，降低被拦概率
    response = requests.get(topics_url, headers=headers)
    # 解析 HTML
    soup = BeautifulSoup(response.content, 'html.parser')

    # 找出所有带 href 的 <a>
    links = soup.find_all("a", href=True)
    # 累积候选病症页 URL
    condition_links = []

    for link in links:
        # 取出原始 href（可能是相对路径 /xxx）
        href = link['href']
        # 相对路径补全为绝对 URL
        if href.startswith("/"):
            href = "https://www.thuisarts.nl" + href
        # 粗规则：同站 https，且路径段够深（split 后 >3），更像具体病症页而非首页
        if href.startswith("https://www.thuisarts.nl/") and len(href.split("/")) > 3:
            condition_links.append(href)

    # set 去重后再转回 list，避免同一链接出现多次
    return list(set(condition_links))


In [16]:
# ========== link_system_prompt：教模型「只保留病症/症状相关链接」的 system 指令 ==========
# 注意：整段 prompt 字符串必须保持英文原样——翻译会改变模型筛选行为

link_system_prompt = """You are an assistant that filters URLs for patient education content. 

Only return links that lead to pages about symptoms, health conditions, treatments, or diseases — for example: pages on 'headache', 'diarrhea', 'stomach pain', 'asthma', etc.

DO NOT return:
- contact pages
- overview/video/image/keuzekaart lists unless they directly link to medical complaints
- navigation or privacy/cookie/social media links

Respond only with full https links in JSON format, like this:
{
  "links": [
    {"type": "symptom or condition page", "url": "https://www.thuisarts.nl/hoofdpijn"},
    {"type": "symptom or condition page", "url": "https://www.thuisarts.nl/buikpijn"}
  ]
}
"""


In [17]:
# 打印 system prompt，方便你肉眼检查「发给模型的筛选规则」长什么样
print(link_system_prompt)


You are an assistant that filters URLs for patient education content. 

Only return links that lead to pages about symptoms, health conditions, treatments, or diseases — for example: pages on 'headache', 'diarrhea', 'stomach pain', 'asthma', etc.

DO NOT return:
- contact pages
- overview/video/image/keuzekaart lists unless they directly link to medical complaints
- navigation or privacy/cookie/social media links

Respond only with full https links in JSON format, like this:
{
  "links": [
    {"type": "symptom or condition page", "url": "https://www.thuisarts.nl/hoofdpijn"},
    {"type": "symptom or condition page", "url": "https://www.thuisarts.nl/buikpijn"}
  ]
}



In [18]:
# ========== 试跑：抓一遍全站候选链接，并转成 [{url: ...}, ...] 结构 ==========

# 调用上面的爬虫函数：得到去重后的病症页 URL 列表
condition_links = get_condition_links_from_topics_page()
# 打印数量；✅ 与英文文案保留，便于对照运行输出
print(f"✅ Found {len(condition_links)} condition pages.")

# 把纯字符串列表映射成字典列表，后续 brochure 流程统一用 item["url"] 取地址
selected_links = [{"url": link} for link in condition_links]


✅ Found 680 condition pages.


In [19]:
# ========== 本地 JSON 缓存：避免每次重跑都对同一 URL 再调一次贵 API ==========

# 再导入一次 json（幂等；与顶部 import 共存无妨）
import json

def load_existing_summaries(filepath="brochure_cache.json"):
    """若缓存文件存在则读成 dict；否则返回空 dict。"""
    # 文件存在才打开，避免首次运行 FileNotFoundError
    if os.path.exists(filepath):
        # encoding=utf-8：荷兰文/英文摘要都安全
        with open(filepath, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}

def save_summaries_to_cache(summaries, filepath="brochure_cache.json"):
    """把 {url: summary, ...} 整表写回磁盘；indent=2 方便人眼查看。"""
    with open(filepath, "w", encoding="utf-8") as f:
        # ensure_ascii=False：非 ASCII 字符原样写入，不转成 \\uXXXX
        json.dump(summaries, f, indent=2, ensure_ascii=False)


In [20]:
# ========== 缩小试跑范围：只取前 10 条链接，控制 API 费用与等待时间 ==========

# 重新抓链接 → 包成 dict → 切片前 10 个（演示够用；要全量可去掉 [:10]）
selected_links = [{"url": link} for link in get_condition_links_from_topics_page()][:10]


In [21]:
# ========== summarize_for_brochure：抓页面 → 拼 prompt → 调 GPT 写英文患者摘要 ==========

# 进程内内存缓存：同一次内核会话里重复 URL 直接命中，少打 API
summary_cache = {}

def summarize_for_brochure(url):
    """对单个病症页生成 brochure 风格英文摘要；命中缓存则直接返回。"""
    # 先查内存缓存
    if url in summary_cache:
        summary = summary_cache[url]
        print(f"✅ [Cached] {url}")
        # 把缓存摘要也打印出来，方便你对照「没重新生成」
        print(f"📄 Summary:\n{summary}\n")  # 👈 this prints the cached summary too
        return summary

    # 未命中：现场抓取并清洗该页
    page = Website(url)

    # few-shot 示例：告诉模型「Title / Summary」长什么样（英文示例保留）
    example = """
Example:

Title: Keelpijn  
Summary: Sore throat is a common symptom, often caused by a virus. It usually goes away on its own within a few days. Drink warm fluids, rest your voice, and take paracetamol if needed. See a doctor if the pain lasts more than a week or gets worse.

Title: Hoofdpijn  
Summary: Headaches can have many causes like stress, fatigue, or dehydration. Most are harmless and go away with rest and fluids. Painkillers like paracetamol can help. If headaches are severe, frequent, or different than usual, contact your GP.
"""

    # 用户 prompt：荷兰文页面内容 → 要求输出英文患者宣传册摘要（字符串内容勿译）
    prompt = f"""
You are a health writer. Based on the Dutch content below, write a clear, short, brochure-style summary in **English** for patients.

Use the format:  
Title: {page.title}  
Summary: <your summary>

Keep it under 100 words, easy to read, friendly, and medically accurate.

{example}

Now use this for:
Title: {page.title}
Content:
{page.text[:3000]}
"""

    # Chat Completions：model / temperature 保持原值，改了会影响文风与成本
    response = openai.chat.completions.create(
        model="gpt-4",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.4
    )

    # 取出助手回复文本并去掉首尾空白
    summary = response.choices[0].message.content.strip()
    # 写入内存缓存，供本次会话后续复用
    summary_cache[url] = summary
    return summary


In [22]:
# ========== build_symptom_brochure：批量摘要 + 磁盘缓存增量保存 ==========

def build_symptom_brochure(links, cache_file="brochure_cache.json"):
    """遍历 links，生成/读取摘要，返回 [{url, summary}, ...] 列表。"""
    # 最终要返回的宣传册条目
    brochure = []
    # 先读磁盘缓存，跨内核重启也能复用
    cached = load_existing_summaries(cache_file)
    print("📄 Building summaries for brochure:\n")

    # enumerate(..., 1)：人读进度从 1 开始
    for i, item in enumerate(links, 1):
        url = item["url"]
        # 磁盘已有 → 直接用，不再打 API
        if url in cached:
            print(f"✅ [Cached] {url}")
            brochure.append({"url": url, "summary": cached[url]})
            continue

        # 新 URL：调用 summarize_for_brochure
        print(f"🔄 [{i}/{len(links)}] Summarizing: {url}")
        try:
            summary = summarize_for_brochure(url)
            print(f"✅ Summary:\n{summary}\n")
            brochure.append({"url": url, "summary": summary})
            # 写回内存中的 cached 字典
            cached[url] = summary  # Save new summary
            # 每成功一条就落盘，中途失败也不丢已完成部分
            save_summaries_to_cache(cached, cache_file)
        except Exception as e:
            # 单条失败不中断整批；占位文案保持英文原样
            print(f"❌ Error summarizing {url}: {e}\n")
            brochure.append({"url": url, "summary": "Error generating summary."})

    return brochure


In [24]:
# ========== 主流程：对 selected_links（前 10 条）生成宣传册摘要列表 ==========
brochure = build_symptom_brochure(selected_links)


📄 Building summaries for brochure:

🔄 [1/10] Summarizing: https://www.thuisarts.nl/sociale-angststoornis
✅ [New] https://www.thuisarts.nl/sociale-angststoornis
📄 Summary:
Title: Social Anxiety Disorder
Summary: Social anxiety disorder, or social phobia, is a fear of what others think of you, often leading to panic attacks. Writing down what happens, your thoughts, and feelings can help manage this fear. Positive thinking can also be beneficial when you're feeling anxious. Discussing your concerns with your GP or practice nurse can be helpful. If there's no improvement or symptoms are severe, treatments such as therapy with a psychologist or anxiety medication may be considered.

✅ Summary:
Title: Social Anxiety Disorder
Summary: Social anxiety disorder, or social phobia, is a fear of what others think of you, often leading to panic attacks. Writing down what happens, your thoughts, and feelings can help manage this fear. Positive thinking can also be beneficial when you're feeling anxi

In [23]:
# ========== 导出：把摘要列表写成纯文本文件，方便分享/打印 ==========

def export_brochure_to_txt(brochure, filepath="brochure_summaries.txt"):
    """将 brochure 列表写入 txt：每条先 URL，再 Summary 正文。"""
    # 空列表直接提示并返回
    if not brochure:
        print("⚠️ No summaries to export.")
        return

    # utf-8 写入，兼容多语言字符
    with open(filepath, "w", encoding="utf-8") as f:
        for item in brochure:
            # .get 带默认值，防止缺字段时崩
            url = item.get("url", "Unknown URL")
            summary = item.get("summary", "No summary available.")
            f.write(f"URL: {url}\n")
            f.write(f"{summary}\n\n")

    print(f"📁 Exported {len(brochure)} summaries to {filepath}")


In [26]:
# 把刚生成的 brochure 导出到默认文件 brochure_summaries.txt
export_brochure_to_txt(brochure)


📁 Exported 10 summaries to brochure_summaries.txt


In [ ]:
# ========== 可选扩展区（原注释「有用」）==========
# 你可以在这里继续试验，例如：
# - 把 selected_links 的 [:10] 调大，生成更完整的患者宣传册
# - 用 link_system_prompt + Chat Completions 再筛一轮「更干净」的病症链接
# - 把 brochure_summaries.txt 改成 Markdown / PDF 排版
# 下面留空单元格可写探索代码


In [ ]:
# （空单元格：需要时在此继续试验；保持为空以免改变原结构）
